Mapa con pueblos

In [9]:
import pandas as pd
import folium

# 1. Cargar los datos
#Cambiar ruta para que no sea local para que sea ejecutable en todo el proyecto

df_municipios = pd.read_csv('../data/datos_municipios.csv') 
df_candidatos = pd.read_csv('../data/candidatos.csv')

# 2. Crear el mapa centrado en Castilla y León
mapa = folium.Map(location=[41.6, -4.7], zoom_start=7)

# 3. Pintar los Helipuertos (Candidatos) en ROJO
for index, row in df_candidatos.iterrows():
    folium.Marker(
        location=[row['latitud'], row['longitud']],
        popup=row['nombre'],
        icon=folium.Icon(color='red', icon='helicopter', prefix='fa')
    ).add_to(mapa)

# 4. Pintar algunos municipios (ej. los grandes) en AZUL
# Filtramos solo los de más de 1000 habitantes para no saturar el mapa al principio
municipios_grandes = df_municipios[df_municipios['poblacion'] > 1000]

for index, row in municipios_grandes.iterrows():
    folium.CircleMarker(
        location=[row['latitud'], row['longitud']],
        radius=3,
        color='blue',
        fill=True,
        fill_color='blue',
        popup=f"{row['municipio']} (Hab: {row['poblacion']})"
    ).add_to(mapa)

# 5. Guardar el mapa
mapa.save('../maps/mapa_inicial_cyl.html')


print("¡Mapa creado! Abre el archivo 'mapa_inicial_cyl.html' en tu navegador.")

¡Mapa creado! Abre el archivo 'mapa_inicial_cyl.html' en tu navegador.


Mapa de calor

In [10]:
import pandas as pd
import folium
from folium.plugins import HeatMap
#Cambiar ruta para que no sea local para que sea ejecutable en todo el proyecto

df_municipios = pd.read_csv('../data/datos_municipios.csv') 
df_candidatos = pd.read_csv('../data/candidatos.csv')

# 2. Crear mapa base
mapa_analisis = folium.Map(location=[41.6, -4.7], zoom_start=7, tiles='CartoDB positron')

# --- TAREA 1: MAPA DE CALOR (DENSIDAD DE POBLACIÓN) ---
# Preparamos los datos: Latitud, Longitud, Peso (Población)
# Normalizamos un poco la población para que el mapa no salga todo rojo
datos_calor = df_municipios[['latitud', 'longitud', 'poblacion']].values.tolist()

# Añadimos la capa de calor
HeatMap(datos_calor, radius=15, blur=20, max_zoom=1).add_to(mapa_analisis)

# --- TAREA 2: RADIOS DE COBERTURA (CÍRCULOS) ---
# Dibujamos un círculo de 40km (40000 metros) alrededor de cada hospital
# Esto simula hasta dónde llega el helicóptero en aprox 15-20 mins.
for index, row in df_candidatos.iterrows():
    # Marcador del hospital
    folium.Marker(
        location=[row['latitud'], row['longitud']],
        popup=row['nombre'],
        icon=folium.Icon(color='red', icon='helicopter', prefix='fa')
    ).add_to(mapa_analisis)
    
    # Círculo de cobertura (Radio en metros)
    folium.Circle(
        location=[row['latitud'], row['longitud']],
        radius=40000,  # 40 km
        color='green',
        fill=True,
        fill_opacity=0.2,
        popup='Cobertura 40km'
    ).add_to(mapa_analisis)

# 3. Guardar
mapa_analisis.save('../maps/mapa_analisis_cobertura2.html')
print("¡Mapa de análisis creado! Ábrelo para ver dónde falta cobertura.")

¡Mapa de análisis creado! Ábrelo para ver dónde falta cobertura.


Mapa final de araña

In [11]:
import pandas as pd
import folium

# 1. Cargar la solución que acabas de generar
try:
    df_solucion = pd.read_csv('..\data\solucion_asignaciones.csv')
except FileNotFoundError:
    print("¡ERROR! Primero tienes que ejecutar el modelo de tus compañeros para generar el CSV.")
    exit()

# 2. Crear mapa base
mapa = folium.Map(location=[41.6, -4.7], zoom_start=7, tiles='CartoDB positron')

# 3. Dibujar las líneas de conexión (Araña)
# Iteramos por cada asignación
for index, row in df_solucion.iterrows():
    
    # Definir color según distancia (Verde: cerca, Rojo: lejos > 45km)
    color_linea = 'green'
    if row['distancia_km'] > 45: # Umbral de 45km (aprox 15-20 min)
        color_linea = 'red'
    elif row['distancia_km'] > 30:
        color_linea = 'orange'
        
    # Dibujar línea
    folium.PolyLine(
        locations=[
            [row['lat_muni'], row['lon_muni']], 
            [row['lat_heli'], row['lon_heli']]
        ],
        color=color_linea,
        weight=0.5,
        opacity=0.4
    ).add_to(mapa)

# 4. Dibujar los Helipuertos ELEGIDOS (Más grandes)
# Agrupamos para no pintar el mismo helipuerto 100 veces
helipuertos_usados = df_solucion[['helipuerto_nombre', 'lat_heli', 'lon_heli']].drop_duplicates()

for index, row in helipuertos_usados.iterrows():
    folium.Marker(
        location=[row['lat_heli'], row['lon_heli']],
        popup=f"BASE ACTIVA: {row['helipuerto_nombre']}",
        icon=folium.Icon(color='blue', icon='helicopter', prefix='fa')
    ).add_to(mapa)

# 5. Guardar
mapa.save('../maps/Mapa_Final_Asignaciones.html')
print("¡Mapa generado! Abre 'Mapa_Final_Asignaciones.html'. Las líneas rojas son pueblos lejos.")

<>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\abela\AppData\Local\Temp\ipykernel_9916\667380292.py:6: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  df_solucion = pd.read_csv('..\data\solucion_asignaciones.csv')


¡Mapa generado! Abre 'Mapa_Final_Asignaciones.html'. Las líneas rojas son pueblos lejos.


Crea graficos

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Cargar los archivos
try:
    df_solucion = pd.read_csv(r'..\data\solucion_asignaciones.csv')
    df_datos = pd.read_csv(r'..\data\datos_municipios.csv')

    print("Columnas df_datos:", df_datos.columns.tolist())
    print("Columnas df_solucion:", df_solucion.columns.tolist())
except FileNotFoundError:
    print("¡ERROR! Faltan archivos. Asegúrate de tener 'solucion_asignaciones.csv' y 'datos_municipios.csv'.")
    exit()

# 2. Cruzar para traer la población
df_completo = pd.merge(
    df_solucion,
    df_datos[['municipio', 'poblacion']],
    on='municipio',
    how='left',
    suffixes=('', '_datos')
)

# Resolver posibles duplicados de 'poblacion'
if 'poblacion_datos' in df_completo.columns:
    df_completo['poblacion'] = df_completo['poblacion_datos']

df_completo['poblacion'] = df_completo['poblacion'].fillna(0)


Columnas df_datos: ['cod_ine', 'municipio', 'provincia', 'poblacion', 'latitud', 'longitud']
Columnas df_solucion: ['municipio_id', 'municipio', 'poblacion', 'lat_muni', 'lon_muni', 'centro_id', 'helipuerto_nombre', 'lat_heli', 'lon_heli', 'tiempo_min', 'distancia_km']
